# Interactive Camera Preview (PyVista)

Explore camera positions with **native mouse rotation/zoom** — no re-render needed
after each camera move.

Unlike the yt-based `interactive_camera.ipynb`, PyVista renders once into a live VTK
window backed by `trame`. Rotate/zoom freely, then run the print-camera cell to copy
the `--cam-position` flag into `plot_geodesics_yt.py` or `movie_geodesics_yt.py`.

**Requirements:** `pip install pyvista[jupyter]` (installs trame + trame-vuetify + trame-vtk)

**Fallback:** set `pv.set_jupyter_backend('static')` to get a static PNG instead.

In [ ]:
import sys
import numpy as np
import pyvista as pv

sys.path.insert(0, '../src')   # import from src/
from data_utils import (
    load_grmhd_density,
    interpolate_to_cartesian,
    load_geodesics,
)
from plot_geodesics_pv import (
    rho_to_pyvista_grid,
    geodesics_to_polydata,
    build_pv_plotter,
)

# Use trame for an interactive 3D window inside Jupyter.
# If trame is not installed, fall back to static PNG:
#   pv.set_jupyter_backend('static')
pv.set_jupyter_backend('trame')

# --- Parameters (edit these) ---
DUMP              = '../data/dump_SANE.h5'
GEODESICS         = '../output/geodesics_str1.h5'
N_GEO             = 50       # number of geodesics to show
R_MAX             = 50.0     # visualisation radius in r_g
RESOLUTION        = 64       # Cartesian grid points per axis (low = fast)
FOLLOW_IDX        = 22       # set to a geodesic index (0-based) to highlight it in gold
DENSITY_RENDER    = False    # set True to add inferno volume rendering of disk density
OPACITY_MULTIPLIER = 100       # higher = more transparent volume (only used when DENSITY_RENDER=True)

In [2]:
# Load and interpolate GRMHD density (run once)
print('Loading GRMHD density ...')
rho, grid_params = load_grmhd_density(DUMP, R_MAX)

print(f'Interpolating to {RESOLUTION}³ Cartesian grid ...')
rho_cart, bbox = interpolate_to_cartesian(rho, grid_params, R_MAX, RESOLUTION)

grid = rho_to_pyvista_grid(rho_cart, bbox, R_MAX)
print('PyVista grid ready:', grid)

Loading GRMHD density ...
Interpolating to 64³ Cartesian grid ...
PyVista grid ready: ImageData (0x16c49c700)
  N Cells:      250047
  N Points:     262144
  X Bounds:     -5.000e+01, 4.844e+01
  Y Bounds:     -5.000e+01, 4.844e+01
  Z Bounds:     -5.000e+01, 4.844e+01
  Dimensions:   64, 64, 64
  Spacing:      1.562e+00, 1.562e+00, 1.562e+00
  N Arrays:     1


In [3]:
# Load geodesic trajectories (run once; skip gracefully if file absent)
r_all = th_all = ph_all = nsteps = idx = None
cyan_lines = gold_lines = None

try:
    r_all, th_all, ph_all, nsteps, idx = load_geodesics(GEODESICS, N_GEO)

    # Ensure the followed geodesic is in the selection
    if FOLLOW_IDX is not None and FOLLOW_IDX not in idx:
        idx = np.append(idx, FOLLOW_IDX)
        print(f'  Added FOLLOW_IDX={FOLLOW_IDX} to selection')

    cyan_lines, gold_lines = geodesics_to_polydata(
        r_all, th_all, ph_all, nsteps, idx, R_MAX, follow_idx=FOLLOW_IDX
    )

    n_cyan = len(idx) if FOLLOW_IDX is None else len(idx) - 1
    n_gold = 1 if FOLLOW_IDX is not None else 0
    print(f'Geodesics: {n_cyan} cyan, {n_gold} gold (FOLLOW_IDX={FOLLOW_IDX})')

except FileNotFoundError:
    print(f'Geodesic file not found ({GEODESICS}); rendering without geodesics.')

  Added FOLLOW_IDX=22 to selection
Geodesics: 50 cyan, 1 gold (FOLLOW_IDX=22)


In [4]:
# Find escaped geodesics — run after the cell above
# Escaped photons reach r >> r_horizon; captured ones stall near r_h ~ 1.35 r_g
R_ESCAPE = 100.0   # r_g threshold

if r_all is not None:
    escaped = []
    for i in range(len(nsteps)):
        ns = nsteps[i]
        if ns < 1:
            continue
        if r_all[i, ns - 1] > R_ESCAPE:
            escaped.append(i)
    print(f'{len(escaped)} escaped geodesics (r_final > {R_ESCAPE} r_g):')
    print(escaped[:20], '...' if len(escaped) > 20 else '')
    print(f'\nSet FOLLOW_IDX to one of these, then re-run the geodesics cell.')
else:
    print('No geodesic data loaded.')

799 escaped geodesics (r_final > 100.0 r_g):
[22, 26, 40, 47, 51, 71, 75, 86, 103, 111, 121, 125, 136, 139, 145, 147, 177, 213, 216, 238] ...

Set FOLLOW_IDX to one of these, then re-run the geodesics cell.


In [5]:
# Build interactive 3D scene
# Rotate/zoom with mouse; left-click drag = rotate, right-click drag = zoom
cam_position = (2.0 * R_MAX, 0.0, 0.0)   # <-- edit starting position (x, y, z) in r_g

plotter = build_pv_plotter(
    grid, cyan_lines, gold_lines, R_MAX, grid_params,
    cam_position=cam_position,
    density_render=DENSITY_RENDER,
    opacity_multiplier=OPACITY_MULTIPLIER,
)
plotter.show()

Widget(value='<iframe src="http://localhost:63502/index.html?ui=P_0x17dbd1c90_0&reconnect=auto" class="pyvista…

In [ ]:
# Print the CLI flag to paste into plot_geodesics_yt.py / movie_geodesics_yt.py
# Run this cell AFTER rotating the scene to the desired view
x, y, z = plotter.camera.position
print(f'--cam-position {x:.4g} {y:.4g} {z:.4g}')

In [ ]:
# Optional: save a screenshot of the current view
# plotter.screenshot('camera_preview.png')
# print('Saved camera_preview.png')